In [34]:
# Importamos las librerias que nos seran de utilidad
import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import mlflow
import mlflow.sklearn
from pathlib import Path


In [35]:

# Configuración de rutas
artifacts_path = './artifacts'
experiment_name = 'default_experiment'

# Crear carpeta de artefactos si no existe
Path(artifacts_path).mkdir(parents=True, exist_ok=True)


In [36]:

# Cargar datos desde archivo CSV
dataset = pd.read_csv("../data/raw/data.csv")
print(dataset.head())

# Separar características y objetivo
X = dataset.drop(columns=['mora'])  
y = dataset['mora']


   mora  atraso  vivienda  edad  dias_lab  exp_sf  nivel_ahorro  ingreso  \
0     0     235  FAMILIAR    30      3748    93.0             5   3500.0   
1     0      18  FAMILIAR    32      4598     9.0            12    900.0   
2     0       0  FAMILIAR    26      5148     8.0             2   2400.0   
3     0       0  FAMILIAR    36      5179    20.0            12   2700.0   
4     0       0  FAMILIAR    46      3960     NaN             1   3100.0   

   linea_sf  deuda_sf  score         zona  clasif_sbs     nivel_educ  
0       NaN      0.00    214         Lima           4  UNIVERSITARIA  
1   1824.67   1933.75    175  La Libertad           1        TECNICA  
2   2797.38    188.29    187         Lima           0  UNIVERSITARIA  
3       NaN      0.00    187       Ancash           0        TECNICA  
4   2000.00  11010.65    189         Lima           0        TECNICA  


In [37]:

# Manejamos los valores faltantes solo en columnas numéricas
numeric_columns = X.select_dtypes(include=['number']).columns
imputer = SimpleImputer(strategy='mean')
X[numeric_columns] = imputer.fit_transform(X[numeric_columns])

# Identificamos las columnas categóricas
categorical_columns = X.select_dtypes(include=['object', 'category']).columns

# Transformamos las columnas categóricas en numéricas
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ], remainder='passthrough'
)

# Dividimos los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Aplicar transformaciones
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)


In [38]:

# Modelos y configuraciones para probar
models = {
    "LogisticRegression": {
        "class": "sklearn.linear_model.LogisticRegression",
        "params": {"random_state": 42}
    },
    "RandomForestClassifier": {
        "class": "sklearn.ensemble.RandomForestClassifier",
        "params": {"n_estimators": 100, "random_state": 42}
    },
    "KNeighborsClassifier": {
        "class": "sklearn.neighbors.KNeighborsClassifier",
        "params": {"n_neighbors": 5}
    },
    "GradientBoostingClassifier": {
        "class": "sklearn.ensemble.GradientBoostingClassifier",
        "params": {"random_state": 42}
    },
    "SVC": {
        "class": "sklearn.svm.SVC",
        "params": {"kernel": "rbf", "C": 1.0, "random_state": 42}
    },
}

best_model = None
best_score = -float('inf')
best_pipeline = None


In [41]:

# Registrar modelos y entrenar
mlflow.set_experiment(experiment_name)
mlflow.end_run()  # End any active run
with mlflow.start_run():
    for model_name, model_config in models.items():
        with mlflow.start_run(nested=True):
            model_class = model_config['class']
            model_params = model_config['params']
            
            # Instanciar el modelo dinámicamente
            module_name, class_name = model_class.rsplit('.', 1)
            module = __import__(module_name, fromlist=[class_name])
            ModelClass = getattr(module, class_name)
            model = ModelClass(**model_params)
            
            # Crear pipeline
            from sklearn.pipeline import Pipeline
            pipeline = Pipeline([
                ("model", model)
            ])
            
            # Entrenar el pipeline
            pipeline.fit(X_train, y_train)
            
            # Evaluar desempeño
            y_pred = pipeline.predict(X_test)
            score = accuracy_score(y_test, y_pred)
            
            # Registrar en mlflow
            mlflow.log_param("model_name", model_name)
            mlflow.log_metric("accuracy", score)
            mlflow.sklearn.log_model(pipeline, f"model_{model_name}")
            
            # Seleccionar el mejor modelo
            if score > best_score:
                best_score = score
                best_model = model_name
                best_pipeline = pickle.dumps(pipeline)

# Guardar el mejor pipeline en artefactos
best_pipeline_path = os.path.join('../artifacts', 'trained_pipeline.pkl')
Path('../artifacts').mkdir(parents=True, exist_ok=True)  # Asegurar que la carpeta exista
with open(best_pipeline_path, 'wb') as f:
    f.write(best_pipeline)

mlflow.log_artifact(best_pipeline_path)  # Registrar el archivo en mlflow

print(f"El mejor modelo fue {best_model} con una precisión de {best_score:.4f}")




/Users/MARINHO/anaconda3/envs/venv-churning-model/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2024/12/19 21:56:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/12/19 21:56:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/12/19 21:56:54 WARNING mlflow.models.model: Model log

El mejor modelo fue RandomForestClassifier con una precisión de 0.8750
